In [4]:
### import necessary libraries
from datetime import datetime
# import geopandas as gpd
# from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt 
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from IPython.display import clear_output
import pandas as pd
from adjustText import adjust_text
# from sklearn.metrics import adjusted_rand_score
# from sklearn.neighbors import NearestNeighbors
# from sklearn.neighbors import KNeighborsClassifier
from datetime import datetime

today = datetime.today().strftime('%Y-%m-%d')

pd.options.display.max_rows = 2000

import matplotlib.pyplot as plt

import matplotlib as mpl


# Ensure text remains editable in SVG

mpl.rcParams['svg.fonttype'] = 'none'  # 'none' = keep text as text objects


# Optional: improve SVG precision

mpl.rcParams['svg.hashsalt'] = ''  # consistent hashes for reproducibility

# warnings.filterwarnings("ignore") 
# sc.logging.print_header()
# sc.set_figure_params(facecolor="white", figsize=(4, 10))
# sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
# plt.rcParams["font.family"] = "Arial"
# sns.set_style("white")

start_time = datetime.now()

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color= ['#009736','#EE2A35',"#3f488aff",
                                                    "#f79a00ff", "#cf1100ff", "#81a5bfff",
                                                    "#f9bd00ff","#547200ff", "#bfd8cdff"]) 

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

In [23]:
from module.misc import sample_name_import

name_dir = 'all-samples-C123'
samples_ids = sample_name_import(name_dir)
samples_ids

dir = 'D:/Xenium'
dir_notebook = 'D:/Jupyter_notebook/Xenium_jupyter_notebook'

## AD

In [ ]:
count = 0

for sample in samples_ids:
    print('Start Sample :', sample)

    df = pd.read_parquet(f'D:/Xenium/{sample}/transcripts.parquet',
                         filters=[("qv",">=",20)]
                         )       

    # df['id_bis'] = "ASSIGNED"
    # df['id_bis'] = df['cell_id'].map({'UNASSIGNED':"UNASSIGNED"})
    # df['id_bis'] = df['id_bis'].fillna('ASSIGNED')
    # ass_unass = df.groupby('id_bis')['feature_name'].value_counts(sort=False)

    
    data = pd.DataFrame({'feature_name': df.feature_name.value_counts().index,'count' : df.feature_name.value_counts()})
    data.sort_index(inplace=True)

    # data["UNASSIGNED"] =  ass_unass["UNASSIGNED"]
    # data['UNASSIGNED'] = data['UNASSIGNED'].fillna(0)
    # data["ASSIGNED"] =  ass_unass["ASSIGNED"]
    # data['ASSIGNED'] = data['ASSIGNED'].fillna(0)
    # data['ratio_un'] = data['UNASSIGNED'] / data['count']
    data['type'] = data['feature_name'].apply(lambda x: x.split('_')[0])

    percentile_threshold:float = 99.5
    threshold = np.percentile(data[(data['type']=="NegControlProbe") or (data['type']=="NegControlCodeword")]['count'].values,percentile_threshold)
    print('threshold = ', threshold)

    data['logfoldovernoise'] = data['count'].apply(lambda x: np.log(x / threshold))
    data_gen_only = data[~(data['feature_name'].str.contains('_'))]
    print('nb of gene under threshold : ', data_gen_only[data_gen_only['logfoldovernoise']<0].shape[0])
    if count == 0:
        set_undernoise = set(data_gen_only[data_gen_only['logfoldovernoise']<0]['feature_name'].values)
    else:
        set_undernoise = set_undernoise.intersection(set(data_gen_only[data_gen_only['logfoldovernoise']<0]['feature_name'].values))

    count += 1
    print("count = ", count)

In [ ]:
set_undernoise

## plot

In [ ]:
type2 = {"DeprecatedCodeword" : "DeprecatedCodeword",
         "NegControlCodeword" : 'NegControlCodeword',
         "NegControlProbe" : "NegControlProbe",
         "UnassignedCodeword" : "UnassignedCodeword"}

data['type2'] = data['type'].map(type2)
data['type2'] = data['type2'].fillna('Gene')

categories = ['Gene', "DeprecatedCodeword","NegControlCodeword","NegControlProbe","UnassignedCodeword"]

def add_jitter(x, scale=0.5):
    return x + np.random.uniform(-scale, scale, size=len(x))

data['Jittered_Category'] = data['type2'].apply(lambda x: categories.index(x))
data['Jittered_Category'] = add_jitter(data['Jittered_Category'])



In [ ]:
# Create a scatter plot with jittered points
plt.scatter(data['Jittered_Category'], data['logfoldovernoise'], alpha=0.5, s=1, color = 'black')
# plt.violinplot(data2, positions = categories)
plt.xticks(ticks=range(len(categories)), labels=categories)
plt.ylabel('LogFold Over Noise')
plt.xticks(rotation=45)
plt.hlines(y=0, xmin=-0.5,xmax=4.5, linestyles='dashed', color = 'red', label='99.5 percentile')
# plt.title('Swarm Plot with Jittered Points')
plt.legend()
plt.show()

In [ ]:
data2 = data[data['type2']=='Gene' or (data['type2']=="NegControlCode")]

In [ ]:
color = {"DeprecatedCodeword" : "blue",
         "NegControlCodeword" : 'green',
         "NegControlProbe" : "red",
         "UnassignedCodeword" : "purple"}
data['color'] = data['type'].map(color)
data['color'] = data['color'].fillna('black')

In [ ]:
plt.scatter(x= data['logfoldovernoise'], y= data['ratio_un'], c=data['color'], cmap="hls", s = 2.5)

In [ ]:
data_no_un = data[~data['type']=="UnassignedCodeword"]

In [ ]:
plt.scatter(x= data_no_un['logfoldovernoise'], y= data_no_un['ratio_un'], c=data_no_un['color'], cmap="hls", s = 2.5)
plt.vlines(x=0, ymin=0, ymax=1, linestyles='dashed', color = 'red')
plt.xlabel('LogFoldOverNoise')
plt.ylabel('Unassigned Ratio')
plt.legend()

## Plot assigned/unassigned transcripts localization

In [ ]:
df = pd.read_parquet('E:/Xenium_SD/SD1-ZT01/transcripts.parquet', filters=[("qv",">=",20), ('feature_name','=','Kif5a')])
df.drop(['overlaps_nucleus','qv', 'codeword_index'],axis=1, inplace=True)
df['id_bis'] = df['cell_id'].map({'UNASSIGNED':"UNASSIGNED"})
df['id_bis'] = df['id_bis'].fillna('ASSIGNED')
df['color'] = df['id_bis'].map({'UNASSIGNED':'red','ASSIGNED':"black"})

In [ ]:
df_crop = df[(df['x_location'].between(3750,4000))&(df['y_location'].between(1300,1500))]
# plt.figure(figsize = (10,6))
plt.scatter(x=df_crop['x_location'],y=df_crop['y_location'], color = df_crop['color'], s = 5)
plt.legend()

In [ ]:
plt.figure(figsize = (10,6))
plt.scatter(x=df['x_location'],y=df['y_location'], color = df['color'], s = 0.01)